# 🛠️ AI Text Processor & 🎙️ TTS Audio Book Generator

**Note:** After "Starting" this loading, any needed interactions (e.g. File uploads) or progress bars are shown BELOW all of the settings!

This delightful tool uses Kokoro TTS and brilliant AI models to spin your ideas into custom audiobooks right in Google Colab—no technical wizardry required!

**Any Input**: Paste text, upload a .txt file, or give the Vision AI an image to describe!

**Smart Processing**: Clean up messy text, use custom prompts, or use the "Generate" & "Recursion" tools to loop a tiny concept into a sprawling story.

**Choose Your Processor**: Pick from strict text-formatting models or wildly creative storytelling ones.

**Beautiful Voices**: Turn your final text into a seamless, high-quality audiobook with a wide selection of voices.

**Safe & Sound**: Auto-save your masterpieces to Google Drive with optional password encryption!

*A Tiny Note*:
The AI loves to make text flow smoothly for audio, so it might slightly tweak your words. Because of these charming quirks, please avoid using this for strict math or highly technical documents where format and punctuation is critical!

*A tinyer note:*
If you need a [HuggingFace token](https://huggingface.co/settings/tokens) to fix "409" errors or unlock a gated model, simply click the little "Key" icon on the left, hit "+ Add new secret", add 'HF_TOKEN', toggle on "Notebook access", and pop your key into the "value" box!

In [ ]:
# @title 🛠️ AI Text Processor & 🎙️ TTS (Text to Speech) Generator 🛠️
# @markdown ### Select your Task and Input Source below:
Task = "Both: Process Text then Generate TTS" # @param ["Text Processor Only", "TTS Generator Only", "Both: Process Text then Generate TTS"]
input_source = "Upload File (.txt or Image)" # @param ["Text Box", "Upload File (.txt or Image)"]
# @markdown **image_prompt:** *(Optional)* If uploading an image, add specific context for things that can't be "Seen" (e.g., "The person in the red dress is named Sarah" or "The story should be about computer games").
image_prompt = "" # @param {type:"string"}
# @markdown **text:** *(Optional)* If input_source is 'Text Box', paste what you wan to be processed here.
text = "" # @param {type:"string"}
# @markdown **higher_quality_outputs:** This is much slower, it adds approximately 25 Mins for the first creation in a session (up from approximately 10-15 mins) and it also adds 5 mins per susiquent run within in the same session! <br />
# @markdown **note**: If using this, I recomend using the "save_to_google_drive" option below to help reduce losses caused by timeouts / crashes!
higher_quality_outputs = False # @param {type:"boolean"}
# @markdown <hr />

# @markdown ### ⚙️ General Settings:
# @markdown **save_to_google_drive:** Automatically save generated outputs (.txt / .wav / .zip) to Google Drive? (Requires login)
save_to_google_drive = False # @param {type:"boolean"}

# @markdown **Save Models to Google Drive:** Save all required files to Google Drive this makes future runs about 1/4 of the time. <br />
# @markdown  ***=== You ALSO need to tick this to USE saved models ===*** <br />
# @markdown ⚠️ **WARNING ON DRIVE SPACE:** Model files are large!! (Standard models ~6GB, "higher_quality_outputs" models ~15-30GB+). Ensure your Google Drive has enough free space before enabling this! <br />
# @markdown *Note: All models are stored cleanly inside the subfolder `MyDrive/Colab_AI_Cache/` to keep your Drive organized. Delete that folder to reclaim space.*
cache_models_to_drive = False # @param {type:"boolean"}

# @markdown **zip_password:** If saving to Drive, encrypt the file(s) with this password. (Leave blank for no encryption)
zip_password = "" # @param {type:"string"}
# @markdown **output_filename:** If using 'Text Box' input name your files here. This uses input filenames if you give it a file.
output_filename = "Processed_File" # @param {type:"string"}
# @markdown <hr />

# @markdown ### 🎙️ TTS Settings (Text to Speech) Settings:
# @markdown **voice:** Select the voice you wish to use. All valid entities and samples can be found [HERE](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX#voicessamples) in the `Voices/Samples` section.<br />
# @markdown *Note:* Voice format is a/b (American/British) f/m (Feminine/Masculine) _name, E.G. af_nova = An American, Feminine voice.
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />

# @markdown ### 🛠️ T2T (Text to Text) Settings:
# @markdown This is the instruction that the "Text processor" uses to process the file. <br />
# @markdown **TextCleaning**: Keeps the current text and removes formatting, page numbers, etc. to make it ready for TTS. E.G. You have copied data out of a PDF and it is awfully formatted. <br />
# @markdown **TextGeneration**: Takes your input and expands it to be more in-depth. E.G. taking "A story about a cat" and turning it into a full story. <br />
# @markdown **Custom**: Use the text box here to give the "Text to Text" engine your own instructions!
prompt_type = "TextGeneration" # @param ["TextCleaning", "TextGeneration", "A monologue in Vlog style", "Custom"]
custom_prompt = "" # @param {type:"string"}

# --- Dynamic Temperature Setting ---
if prompt_type == "TextCleaning":
    gen_temp = 0.1 # Low temp for strict adherence and formatting
else:
    gen_temp = 0.75 # Higher temp for creativity in storytelling

if prompt_type == "TextCleaning":
    system_prompt = (
        "You are an expert audio-text preparer. Your task is to process this text "
        "so it reads smoothly for Text-to-Speech processing. 1. Remove random line breaks "
        "to reconstruct proper flowing paragraphs. 2. Fix broken hyphenations (e.g., "
        "'para- graph' becomes 'paragraph'). 3. Normalize spacing by removing extra spaces "
        "or tabs. 4. Delete inline headers, footers, page numbers, and stray isolated numbers. "
        "5. DO NOT rewrite, summarize, or change the author's original words. Output ONLY the "
        "processed text with no conversational filler."
    )
elif prompt_type == "TextGeneration":
    system_prompt = (
        "You are an award-winning novelist and master storyteller. Your task is to write a compelling, "
        "deeply immersive story based on the provided text or concept. "
        "1. Structure: Build a complete narrative arc with a captivating hook, escalating tension, "
        "a distinct climax, and a resonant resolution. "
        "2. World & Character: Craft multi-dimensional characters with distinct voices and internal "
        "motivations. Anchor them in a vivid, lived-in setting using visceral sensory details. Apply the "
        "'show, don't tell' principle. "
        "3. Pacing & Depth: Expand the core concept substantially to ensure a lengthy, detailed read. "
        "Use varied sentence structures to control the pacing naturally. "
        "4. Tone: Establish a consistent atmosphere that aligns perfectly with the input's genre, "
        "prioritizing emotional authenticity. "
        "5. Constraints: DO NOT include titles, introductions, meta-commentary, or conversational filler. "
        "Output absolutely nothing but the story itself."
    )
elif prompt_type == "A monologue in Vlog style":
    system_prompt = (
        "You are recording a raw, totally unscripted, direct-to-camera vlog on your phone. "
        "CRITICAL DIRECTIVE: If the prompt includes an image description or character details, YOU ARE THAT EXACT CHARACTER. You must fully adopt their implied background and current situation as your own reality. "
        "1. Action-Driven First-Person Narration: Ground the monologue entirely in forward momentum. Speak extensively in the first person ('I', 'me', 'my'). Instead of describing your surroundings or appearance, narrate the actions you are actively taking, the decisions you are making, and the tasks you are trying to accomplish right now. Keep the narrative moving forward—focus on what you are doing next, not what you are looking at. "
        "2. Total Immersion & Progression: Treat the provided context as an active, unfolding event. Don't waste time painting a visual picture of the room; interact with it. React to the situation by deciding on your next move, confronting an issue, or rushing to get something done. Let the viewer infer the setting through your active engagement with it. "
        "3. Unscripted & Conversational (TTS Optimized): Write specifically for a Text-to-Speech engine. It must sound 100% human and unedited. Include natural vocal disfluencies: mid-sentence tangents, self-corrections (e.g., 'Wait, actually no...'), trailing thoughts, and natural fillers ('like', 'I mean', 'you know', 'look'). Do not sound like a written essay; sound like someone actively thinking out loud while busy doing other things. "
        "4. Absolute Ban on Non-Speech Elements: DO NOT output any AI filler or meta-commentary (e.g., 'Here is your monologue', 'Let's begin'). DO NOT include stage directions, action tags, emojis, or formatting (no *sighs*, no [looks at camera], no bold text, no headers). Output ONLY the exact, raw spoken words to be fed directly into the audio generator. Start speaking immediately on the very first word."
    )
elif prompt_type == "Custom":
    system_prompt = custom_prompt

# @markdown **recursion_loops:** How many times should the AI take its own output and feed it back into itself to expand/continue the text? (1 = process input once. 2+ = continue extending the text). <br />
# @markdown **Note**: Leave this as "1" for TextCleaning.
recursion_loops = 1 # @param {type:"slider", min:1, max:20, step:1}

# @markdown **repetition_penalty (UI Default):** Applied to the initial generation. (1.0 is no penalty). For recursive loops, this script will safely lower this to prevent gibberish.
repetition_penalty = 1.15 # @param {type:"slider", min:1.0, max:2.0, step:0.05}

# @markdown ### Text Processing Model Selector:
model_choice = "Qwen/Qwen2.5-3B-Instruct" # @param ["Qwen/Qwen2.5-3B-Instruct", "dphn/Dolphin3.0-Qwen2.5-3b", "dphn/Dolphin3.0-Llama3.2-3B"] {allow-input: true}


# ==============================================================================
# 🚀 MAIN SCRIPT EXECUTION
# ==============================================================================
import os
import sys
# Forces sequential downloads, stopping disk I/O bottlenecks
os.environ["HF_HUB_DISABLE_XET"] = "1"
import subprocess
import shutil
import re
import urllib.request
import zipfile
import gc
import threading
import numpy as np
from IPython.display import Audio, display, clear_output
from google.colab import files

# --- 1. MODEL CACHING MOUNT (MUST HAPPEN BEFORE DOWNLOADS) ---
if cache_models_to_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("📂 Mounting Google Drive for Model Caching, Please login using the popup window...")
        drive.mount('/content/drive')

    drive_cache_dir = "/content/drive/MyDrive/Colab_AI_Cache/hf_cache"
    os.makedirs(drive_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = drive_cache_dir
    print(f"⚡ Google Drive Model Caching active! Storage directory: {drive_cache_dir}")

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN secret exists but is empty.")
    os.environ["HF_TOKEN"] = hf_token
    print("🔑 HF_TOKEN successfully loaded from Colab Secrets!")
except userdata.SecretNotFoundError:
    print("⚠️ 'HF_TOKEN' secret not found in Colab settings. Running unauthenticated.")
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"
except Exception as e:
    print(f"⚠️ Unexpected error loading HF_TOKEN: {e}. Running unauthenticated.")
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"

def flush_memory():
    """Forces garbage collection and clears CUDA cache"""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass

# ==============================================================================
# ⚡ BACKGROUND MODEL PRE-DOWNLOADER
# ==============================================================================
bg_threads = {}

def bg_download_hf(repo_id):
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id=repo_id,
            allow_patterns=["*.safetensors", "*.json", "*.model", "*.txt", "*.onnx", "tokenizer*"]
        )
    except Exception:
        pass

def bg_download_tts():
    model_file, voices_file = "kokoro-v1.0.onnx", "voices-v1.0.bin"
    if not os.path.exists(model_file):
        urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx", model_file)
    if not os.path.exists(voices_file):
        urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin", voices_file)

print("⚡ Launching background model pre-downloaders...")

actual_model_choice = model_choice
if higher_quality_outputs:
    if model_choice == "Qwen/Qwen2.5-3B-Instruct": actual_model_choice = "Qwen/Qwen2.5-14B-Instruct"
    elif model_choice == "dphn/Dolphin3.0-Qwen2.5-3b": actual_model_choice = "cognitivecomputations/dolphin-2.9.2-qwen2-7b"
    elif model_choice == "dphn/Dolphin3.0-Llama3.2-3B": actual_model_choice = "dphn/Dolphin3.0-Llama3.1-8B"

if "Text Processor" in Task or "Both" in Task:
    t_text = threading.Thread(target=bg_download_hf, args=(actual_model_choice,), daemon=True)
    t_text.start()
    bg_threads['text'] = t_text

if "TTS Generator" in Task or "Both" in Task:
    t_tts = threading.Thread(target=bg_download_tts, daemon=True)
    t_tts.start()
    bg_threads['tts'] = t_tts

if input_source == "Upload File (.txt or Image)":
    t_vision = threading.Thread(target=bg_download_hf, args=("llava-hf/llava-onevision-qwen2-0.5b-ov-hf",), daemon=True)
    t_vision.start()
    bg_threads['vision'] = t_vision

# --- 2. SAVE OUTPUTS MOUNT (HAPPENS WHILE MODELS DOWNLOAD IN BACKGROUND) ---
if save_to_google_drive and not cache_models_to_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("📂 Mounting Google Drive for output saving. (Models are downloading in the background!)")
        drive.mount('/content/drive')

if zip_password.strip():
    try:
        subprocess.run(["7z"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        print("📦 Installing 7-zip for encryption...")
        subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full", shell=True, check=True)

# ==============================================================================
# 🧩 CORE FUNCTIONS
# ==============================================================================

def extract_input_data():
    raw_input = ""
    base_name = output_filename
    is_txt_upload = False # Flag to track if the user uploaded a text file

    if input_source == "Upload File (.txt or Image)":
        print("📂 Awaiting file upload... (Models are pre-downloading in background)")
        uploaded = files.upload()
        if not uploaded:
            print("❌ No file uploaded. Execution stopped.")
            sys.exit()

        original_filename = list(uploaded.keys())[0]
        base_name = os.path.splitext(original_filename)[0]
        ext = os.path.splitext(original_filename)[1].lower()
        image_extensions = ['.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif', '.tiff']

        if ext in image_extensions:
            print(f"🖼️ Detected image file '{original_filename}'. Analyzing with Vision model...")

            if 'vision' in bg_threads:
                bg_threads['vision'].join()

            try:
                from PIL import Image
                import torch
                from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
            except ImportError:
                print("📦 Installing required image processing libraries... (Silently)")
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "Pillow", "accelerate"], check=True, capture_output=True)
                from PIL import Image
                import torch
                from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration

            image = Image.open(original_filename).convert('RGB')
            device_name = "cuda" if torch.cuda.is_available() else "cpu"
            print("⏳ Loading Vision Model (LLaVA-OneVision-Qwen2-0.5B)...")

            model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
            processor = AutoProcessor.from_pretrained(model_id)
            vision_model = LlavaOnevisionForConditionalGeneration.from_pretrained(
                model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True
            ).to(device_name)

            print("👁️ Extracting visual details...")
            combined_prompt = f"The subject of this image is: {image_prompt.strip()}. Describe this image in detail, capturing all visual elements." if image_prompt.strip() else "Describe this image in detail. Be thorough and capture all the visual elements."
            conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": combined_prompt}]}]
            prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(device_name, torch.float16)

            out = vision_model.generate(**inputs, max_new_tokens=300)
            generated_ids = out[0][inputs.input_ids.shape[1]:]
            description = processor.decode(generated_ids, skip_special_tokens=True).strip()

            raw_input = f"Context: {image_prompt.strip()}\n\nImage Description:\n{description}" if image_prompt.strip() else description
            print(f"📝 Image Description generated:\n{raw_input}")

            del vision_model
            del processor
            flush_memory()

        else:
            is_txt_upload = True # Flagging that a text document was uploaded
            try:
                raw_input = uploaded[original_filename].decode('utf-8')
                print(f"✅ Loaded text file '{original_filename}' successfully.")
            except UnicodeDecodeError:
                try:
                    raw_input = uploaded[original_filename].decode('latin-1')
                    print(f"✅ Loaded text file '{original_filename}' successfully (latin-1 encoding).")
                except Exception:
                    print(f"❌ Error: Could not decode text file '{original_filename}'.")
                    sys.exit()
    else:
        raw_input = text

    if not raw_input.strip():
        print("⚠️ No input detected. Please provide text or upload a file.")
        sys.exit()

    return raw_input, base_name, is_txt_upload

def chunk_text_semantically(input_text, max_chars=2500):
    sentences = re.split(r'(?<=[.!?\n])\s+', input_text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

def process_text_pipeline(input_text, base_name, is_txt_upload):
    print("\n" + "="*50)
    print("🖨️ STARTING AI TEXT PROCESSOR...")
    print("="*50)

    if 'text' in bg_threads:
        print("⏳ Waiting for background model download to finish...")
        bg_threads['text'].join()

    try:
        import transformers
    except ImportError:
        print("📦 Installing required base libraries... (Silently)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True, capture_output=True)

    if higher_quality_outputs:
        try:
            import bitsandbytes
        except ImportError:
            print("📦 Installing bitsandbytes for 4-bit quantization (High Quality Mode)...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True, capture_output=True)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f"⏳ Loading {actual_model_choice} into GPU...")
    tokenizer = AutoTokenizer.from_pretrained(actual_model_choice)

    if higher_quality_outputs:
        print("🚀 'Higher Quality Outputs' enabled. Loading large model in optimized 4-bit mode...")
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(actual_model_choice, device_map="auto", quantization_config=quantization_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(actual_model_choice, torch_dtype=torch.float16, device_map="auto")

    def process_chunk(messages, current_temp, current_rep):
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=2000, temperature=current_temp,
            repetition_penalty=current_rep, do_sample=True,
        )
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Apply "_processed" ONLY if it's a text upload
    suffix = "_processed" if is_txt_upload else ""
    final_txt_filename = f"{base_name}{suffix}.txt"

    chunks = chunk_text_semantically(input_text, max_chars=2500)
    print(f"🧩 Document semantically split into {len(chunks)} chunks.")

    with open(final_txt_filename, "w", encoding="utf-8") as f:
        f.write("")

    full_generated_story = ""
    for i, chunk in enumerate(tqdm(chunks, desc="🖨️ Processing Sections")):
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Please process the following text:\n\n{chunk}"}]
        processed_chunk = process_chunk(messages, current_temp=gen_temp, current_rep=repetition_penalty)
        full_generated_story += processed_chunk + "\n\n"
        with open(final_txt_filename, "a", encoding="utf-8") as f:
            f.write(processed_chunk + "\n\n")

    if recursion_loops > 1:
        print(f"\n🔄 Starting recursion to expand the text ({recursion_loops - 1} additional passes)...")
        recursion_rep_penalty = 1.05
        recursion_temp = max(0.1, gen_temp - 0.05) if prompt_type != "TextCleaning" else gen_temp

        for loop in tqdm(range(recursion_loops - 1), desc="🔄 Recursion Loops"):
            sliding_context = full_generated_story[-3000:].strip() if len(full_generated_story) > 3000 else full_generated_story.strip()
            recursion_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Here is the latest part of the text/story:\n\n...\n{sliding_context}\n\nPlease continue exactly from where it left off. Maintain the same style, tone, and formatting. Do not repeat what was already written or loop backwards. Output ONLY the new continuation without any conversational commentary or titles."}
            ]
            continuation = process_chunk(recursion_messages, current_temp=recursion_temp, current_rep=recursion_rep_penalty)
            full_generated_story += continuation + "\n\n"
            with open(final_txt_filename, "a", encoding="utf-8") as f:
                f.write(continuation + "\n\n")

    del model
    del tokenizer
    flush_memory()
    return final_txt_filename, full_generated_story

def generate_tts_pipeline(input_text, base_name, is_txt_upload):
    print("\n" + "="*50)
    print("🎙️ STARTING KOKORO TTS GENERATOR (ONNX)...")
    print("="*50)

    import logging
    import warnings

    # Forcefully silence the phonemizer logger
    phonemizer_logger = logging.getLogger('phonemizer')
    phonemizer_logger.setLevel(logging.CRITICAL)
    phonemizer_logger.propagate = False

    # Catch standard python warnings just in case
    warnings.filterwarnings("ignore", message=".*words count mismatch.*")

    if 'tts' in bg_threads:
        bg_threads['tts'].join()

    try:
        from kokoro_onnx import Kokoro
        import soundfile as sf

    except ImportError:
        print("📦 Installing Python 3.13 compatible Kokoro ONNX engine...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro-onnx", "soundfile"], check=True)
        from kokoro_onnx import Kokoro
        import soundfile as sf

    model_file, voices_file = "kokoro-v1.0.onnx", "voices-v1.0.bin"
    if not os.path.exists(model_file):
        print("📥 Downloading Kokoro ONNX model (~300MB)...")
        urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx", model_file)
    if not os.path.exists(voices_file):
        print("📥 Downloading Kokoro voice profiles (~15MB)...")
        urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin", voices_file)

    # Apply "_processed" ONLY if it's a text upload
    suffix = "_processed" if is_txt_upload else ""
    final_wav_filename = f"{base_name}{suffix}.wav"

    kokoro = Kokoro(model_file, voices_file)
    lang = {'a': 'en-us', 'b': 'en-gb', 'f': 'fr-fr', 'e': 'es', 'j': 'ja', 'z': 'zh'}.get(voice[0], 'en-us')

    print(f"Generating speech for voice: {voice}...")
    text_chunks = [chunk.strip() for chunk in re.split(r'(?<=[.!?\n])\s+', input_text) if chunk.strip()]
    audio_chunks = []
    dynamic_sample_rate = 24000

    for chunk_text in tqdm(text_chunks, desc="🗣️ Rendering Audio"):
        samples, dynamic_sample_rate = kokoro.create(chunk_text, voice=voice, speed=1.0, lang=lang)
        audio_chunks.append(samples)

    if audio_chunks:
        final_audio = np.concatenate(audio_chunks)
        sf.write(final_wav_filename, final_audio, dynamic_sample_rate)
        display(Audio(final_wav_filename, autoplay=False))

    del kokoro
    flush_memory()
    return final_wav_filename

# ==============================================================================
# 🚀 EXECUTION & FILE MANAGEMENT
# ==============================================================================

# Now receiving the text upload flag here
current_text, base_name, is_txt_upload = extract_input_data()
generated_files = []

if "Text Processor" in Task or "Both" in Task:
    # Passing the text upload flag into the pipelines
    final_txt, current_text = process_text_pipeline(current_text, base_name, is_txt_upload)
    generated_files.append(final_txt)

if "TTS Generator" in Task or "Both" in Task:
    final_wav = generate_tts_pipeline(current_text, base_name, is_txt_upload)
    if final_wav: generated_files.append(final_wav)

print("\n" + "="*50)
print("💾 PREPARING DOWNLOADS & SAVES...")
print("="*50)

if not generated_files:
    print("❌ No files were generated.")
    sys.exit()

if zip_password.strip():
    export_archive = f"{base_name}_outputs.7z"
    print(f"🔒 Encrypting files into {export_archive}...")
    subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", export_archive] + generated_files, stdout=subprocess.DEVNULL)

    if save_to_google_drive:
        drive_path = f"/content/drive/MyDrive/{export_archive}"
        shutil.copy(export_archive, drive_path)
        print(f"✅ Saved encrypted archive to Drive: {drive_path}")

    print("📥 Triggering download...")
    files.download(export_archive)

else:
    if len(generated_files) > 1:
        export_archive = f"{base_name}_outputs.zip"
        print(f"📦 Zipping files together into {export_archive}...")
        with zipfile.ZipFile(export_archive, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in generated_files:
                zipf.write(f)

        if save_to_google_drive:
            drive_path = f"/content/drive/MyDrive/{export_archive}"
            shutil.copy(export_archive, drive_path)
            print(f"✅ Saved zip archive to Drive: {drive_path}")

        print("📥 Triggering download...")
        files.download(export_archive)
    else:
        single_file = generated_files[0]
        if save_to_google_drive:
            drive_path = f"/content/drive/MyDrive/{single_file}"
            shutil.copy(single_file, drive_path)
            print(f"✅ Saved {single_file} to Drive: {drive_path}")

        print("📥 Triggering download...")
        files.download(single_file)

print("\n🎉 All Tasks Completed Successfully!")